# 1. Exploring the LendingClub data

A first look at the raw data.

### Setup

In [18]:
from pathlib import Path
import pandas as pd

# cwd stands for current working directory
ROOT = Path.cwd().parent                      # project folder (this notebook is in notebooks/)
RAW_FILE = ROOT / "data" / "raw" / "accepted_2007_to_2018Q4.csv.gz"

In [19]:
data = pd.read_csv(RAW_FILE, nrows = 5) # the first 5 loans, all columns
data.head().T

,0,1,2,3,4
id,68407277,68355089,68341763,66310712,68476807
member_id,NaN,NaN,NaN,NaN,NaN
loan_amnt,3600.0,24700.0,20000.0,35000.0,10400.0
funded_amnt,3600.0,24700.0,20000.0,35000.0,10400.0
funded_amnt_inv,3600.0,24700.0,20000.0,35000.0,10400.0
...,...,...,...,...,...
settlement_status,NaN,NaN,NaN,NaN,NaN
settlement_date,NaN,NaN,NaN,NaN,NaN
settlement_amount,NaN,NaN,NaN,NaN,NaN
settlement_percentage,NaN,NaN,NaN,NaN,NaN


In [15]:
date_cols = ["issue_d", "earliest_cr_line", "last_pymnt_d", "next_pymnt_d", "last_credit_pull_d"]
data[date_cols]

,issue_d,earliest_cr_line,last_pymnt_d,next_pymnt_d,last_credit_pull_d
0,Dec-2015,Aug-2003,Jan-2019,NaN,Mar-2019
1,Dec-2015,Dec-1999,Jun-2016,NaN,Mar-2019
2,Dec-2015,Aug-2000,Jun-2017,NaN,Mar-2019
3,Dec-2015,Sep-2008,Feb-2019,Apr-2019,Mar-2019
4,Dec-2015,Jun-1998,Jul-2016,NaN,Mar-2018


In [20]:
n_rows = len(pd.read_csv(RAW_FILE, usecols=["id"], dtype=str))   # count every row (reads only the id column)

print(f"Rows: {n_rows:,}")
print(f"Columns: {data.shape[1]}")

Rows: 2,260,701
Columns: 151


In [8]:
# Read 1,000 loans so pandas can tell each column's type reliably (5 rows is too few)
sample = pd.read_csv(RAW_FILE, nrows=1000, low_memory=False)

types = sample.dtypes.astype(str)
print("Columns by type:")
print(types.value_counts().to_string(), "\n")


for t in types.unique():
    cols = types[types == t].index.tolist()
    print(f"{t} ({len(cols)} columns):")
    print("  " + ", ".join(cols), "\n")

Columns by type:
float64    115
str         35
int64        1 

int64 (1 columns):
  id 

float64 (115 columns):
  member_id, loan_amnt, funded_amnt, funded_amnt_inv, int_rate, installment, annual_inc, desc, dti, delinq_2yrs, fico_range_low, fico_range_high, inq_last_6mths, mths_since_last_delinq, mths_since_last_record, open_acc, pub_rec, revol_bal, revol_util, total_acc, out_prncp, out_prncp_inv, total_pymnt, total_pymnt_inv, total_rec_prncp, total_rec_int, total_rec_late_fee, recoveries, collection_recovery_fee, last_pymnt_amnt, last_fico_range_high, last_fico_range_low, collections_12_mths_ex_med, mths_since_last_major_derog, policy_code, annual_inc_joint, dti_joint, acc_now_delinq, tot_coll_amt, tot_cur_bal, open_acc_6m, open_act_il, open_il_12m, open_il_24m, mths_since_rcnt_il, total_bal_il, il_util, open_rv_12m, open_rv_24m, max_bal_bc, all_util, total_rev_hi_lim, inq_fi, total_cu_tl, inq_last_12m, acc_open_past_24mths, avg_cur_bal, bc_open_to_buy, bc_util, chargeoff_within_12_m

### 1.2 Split by loan term

LendingClub offers 36-month and 60-month loans, and the data records each loan's status as of December 2018. A loan only has a known outcome if it was due to finish by then, so each term keeps only the issue years in which every loan had time to finish (Methodology Section 2.2):

- **36-month loans** issued 2007–2015;
- **60-month loans** issued 2010–2013 (60-month loans were introduced in 2010).

Only the columns listed in Methodology Section 3 are loaded.

In [22]:
# Columns to load: id, date, outcome, term and the candidate features (Methodology Section 3)
COLUMNS = [
    "id", "issue_d", "loan_status", "term",
    "loan_amnt", "purpose",
    "annual_inc", "verification_status", "emp_length", "home_ownership", "dti", "addr_state",
    "fico_range_low", "fico_range_high", "earliest_cr_line", "delinq_2yrs", "inq_last_6mths",
    "open_acc", "total_acc", "pub_rec", "pub_rec_bankruptcies", "revol_bal", "revol_util", "mort_acc", "funded_amnt", "total_pymnt",
]
df = pd.read_csv(RAW_FILE, usecols=COLUMNS, low_memory=False) #low_memory=False is needed to avoid a warning about mixed types in columns

# Convert issue date from text ("Dec-2015") to a real date; non-dates (summary lines) become NaT
df["issue_d"] = pd.to_datetime(df["issue_d"], format="%b-%Y", errors="coerce")
# Drop rows with missing issue date
df = df.dropna(subset=["issue_d"])
# Add a column for the year in which the loan was issued
df["issue_year"] = df["issue_d"].dt.year
# Strip whitespace from the term column, which has values like " 36 months" and " 60 months"
df["term"] = df["term"].str.strip()

# Split by term, keeping only the years in which every loan had time to finish by December 2018
loans_36 = df[(df["term"] == "36 months") & df["issue_year"].between(2007, 2015)].copy() # copy makes a new DataFrame instead of the original df
loans_60 = df[(df["term"] == "60 months") & df["issue_year"].between(2010, 2013)].copy()

print(f"36-month loans, issued 2007–2015: {len(loans_36):,}")
print(f"60-month loans, issued 2010–2013: {len(loans_60):,}")

36-month loans, issued 2007–2015: 621,169
60-month loans, issued 2010–2013: 55,290


### 1.3 Loan status

What had happened to each loan by December 2018, in each group.

In [23]:
status_counts = pd.DataFrame({
    "36-month loans": loans_36["loan_status"].value_counts(),
    "60-month loans": loans_60["loan_status"].value_counts(),
}).fillna(0).astype(int)
# if a loan status is missing in one of the two DataFrames, fill it with 0 and convert to int

status_counts

,36-month loans,60-month loans
loan_status,,
Charged Off,85955,13830
Current,72,6
Does not meet the credit policy. Status:Charged Off,649,112
Does not meet the credit policy. Status:Fully Paid,1789,199
Fully Paid,532629,41139
In Grace Period,9,1
Late (16-30 days),5,0
Late (31-120 days),61,3


### 1.4 The target

Each group keeps only loans with a final outcome, issued under LendingClub's standard credit policy, and the target is set to 0 for Fully Paid and 1 for Charged Off or Default. Loans issued under the old credit policy, and the few still running in December 2018, are removed.

In [24]:
# Target from loan_status (Methodology Section 4): 0 = repaid, 1 = defaulted
TARGET = {"Fully Paid": 0, "Charged Off": 1, "Default": 1}

def make_target(loans, name):
    # Loans issued under LendingClub's older rules
    old_policy = loans["loan_status"].str.startswith("Does not meet the credit policy")
    # Loans with no final outcome yet (Current, Late, In Grace Period), ~ means "not"
    unfinished = ~loans["loan_status"].isin(TARGET.keys()) & ~old_policy

    # Keep only finished loans; this drops both groups above
    kept = loans[loans["loan_status"].isin(TARGET.keys())].copy()
    kept["target"] = kept["loan_status"].map(TARGET)   # status -> 0 or 1

    print(f"{name}: removed {old_policy.sum():,} old-policy and {unfinished.sum():,} still running, kept {len(kept):,}")
    return kept

loans_36 = make_target(loans_36, "36-month loans")
loans_60 = make_target(loans_60, "60-month loans")

36-month loans: removed 2,438 old-policy and 147 still running, kept 618,584
60-month loans: removed 311 old-policy and 10 still running, kept 54,969


### 1.5 Preparing the features

Some columns need reshaping before the model can use them (Methodology Section 5). For example, the FICO range becomes a single score. Each new value only uses information from its own loan and we do this before splitting into train and test.

| New feature | Made from | How |
|---|---|---|
| `fico` | `fico_range_low`, `fico_range_high` | average of the two ends of the FICO range |
| `credit_history_years` | `earliest_cr_line`, `issue_d` | years from the first credit account to the loan's issue date |
| `loan_to_income` | `loan_amnt`, `annual_inc` | loan amount divided by annual income |
| `emp_length_years` | `emp_length` | `< 1 year` becomes 0, `10+ years` becomes 10, missing stays missing |

Once the new features are built, the columns they replace are dropped: the two FICO ends, the first credit date, the job length text and the loan amount. Annual income stays, as a feature in its own right.

In [25]:
EMP_LENGTH_YEARS = {
    "< 1 year": 0, "1 year": 1, "2 years": 2, "3 years": 3, "4 years": 4,
    "5 years": 5, "6 years": 6, "7 years": 7, "8 years": 8, "9 years": 9,
    "10+ years": 10,
}

def prepare_features(loans):
    out = loans.copy()

    # FICO score: the middle of the 4-point range
    out["fico"] = (out["fico_range_low"] + out["fico_range_high"]) / 2

    # Length of credit history when the loan was issued, in years
    first_credit = pd.to_datetime(out["earliest_cr_line"], format="%b-%Y", errors="coerce")
    out["credit_history_years"] = (out["issue_d"] - first_credit).dt.days / 365.25

    # Size of the loan relative to yearly income (an income of 0 is treated as missing)
    out["loan_to_income"] = out["loan_amnt"] / out["annual_inc"].where(out["annual_inc"] > 0)

    # Years in current job as a number; values not in the dictionary become missing
    out["emp_length_years"] = out["emp_length"].map(EMP_LENGTH_YEARS)

    # Drop the raw columns that have been replaced
    return out.drop(columns=["fico_range_low", "fico_range_high", "earliest_cr_line",
                             "loan_amnt", "emp_length"])

loans_36 = prepare_features(loans_36)
loans_60 = prepare_features(loans_60)

loans_36[["fico", "credit_history_years", "loan_to_income", "emp_length_years"]].describe()

,fico,credit_history_years,loan_to_income,emp_length_years
count,618584.000000,618584.000000,618582.000000,582445.000000
mean,697.113662,16.084328,0.196283,5.849055
std,31.148584,7.591598,0.106346,3.669516
min,627.000000,2.997947,0.000309,0.000000
25%,672.000000,10.915811,0.114286,2.000000
50%,692.000000,14.581793,0.180000,6.000000
75%,712.000000,19.915127,0.266667,10.000000
max,847.500000,71.000684,10.000000,10.000000


### Missing values by year

How many loans are missing each feature, year by year.

In [28]:
# Columns with at least one missing value
cols = [c for c in loans_36.columns if loans_36[c].isna().any()]

# Missing values per column, for each issue year
missing = loans_36[cols].isna().groupby(loans_36["issue_year"]).sum()

# Number of loans each year
missing["loans that year"] = loans_36.groupby("issue_year").size()

# Add a bottom row with the totals across all years
missing.loc["Total"] = missing.sum()

print("36-month loans: number of loans missing per year")
missing

36-month loans: number of loans missing per year


,dti,revol_util,mort_acc,pub_rec_bankruptcies,loan_to_income,emp_length_years,loans that year
issue_year,,,,,,,
2007,0,0,251,244,0,0,251
2008,0,8,1562,450,0,0,1562
2009,0,14,4716,3,0,0,4716
2010,0,18,8466,0,0,254,8466
2011,0,7,14101,0,0,555,14101
2012,0,41,6025,0,0,1729,43470
2013,0,56,0,0,0,4995,100422
2014,0,77,0,0,0,9646,162570
2015,2,126,0,0,2,18960,283026


Similarly we do for the 60-month loans

In [9]:
cols = [c for c in loans_60.columns if loans_60[c].isna().any()]

missing = loans_60[cols].isna().groupby(loans_60["issue_year"]).sum()
missing["loans that year"] = loans_60.groupby("issue_year").size()
missing.loc["Total"] = missing.sum()

print("60-month loans: number of loans missing per year")
missing

60-month loans: number of loans missing per year


,revol_util,mort_acc,emp_length_years,loans that year
issue_year,,,,
2010,1,3070,65,3070
2011,2,7620,204,7620
2012,6,1470,196,9897
2013,22,0,967,34382
Total,31,12160,1432,54969


### Dropping two columns

The tables show two columns that LendingClub did not record for its early loans:

- `mort_acc` (number of mortgage accounts) is empty for every loan issued before 2012. If we kept it, the model would learn that "missing" means "an old loan" and it can affect how it understands missing values in the test sets.
- `pub_rec_bankruptcies` (number of bankruptcies) is empty for almost every loan in 2007 and for about a third in 2008. This feature is in fact not needed because bankruptcies are already counted in `pub_rec` (all public records).

Both are dropped. The other columns have only a few missing values, spread evenly across the years. That looks like borrowers leaving a question blank rather than a column that wasn't recorded, so those columns are kept.

In [29]:
DROP = ["mort_acc", "pub_rec_bankruptcies"]

loans_36 = loans_36.drop(columns=DROP)
loans_60 = loans_60.drop(columns=DROP, errors="ignore")

print("Dropped:", DROP)
print(f"36-month loans: {loans_36.shape[1]} columns left")
print(f"60-month loans: {loans_60.shape[1]} columns left")

Dropped: ['mort_acc', 'pub_rec_bankruptcies']
36-month loans: 25 columns left
60-month loans: 25 columns left


### 1.6 Train and test sets

Within each group, loans are split by issue date: the model is built on older loans (the **train set**) and tested on newer loans it has never seen (the **test set**), as a bank would do.

| Group | Train | Test |
|---|---|---|
| 36-month loans | issued 2007–2014 | issued 2015 |
| 60-month loans | issued January 2010 – June 2013 | issued July–December 2013 |

In [30]:
test_start_36 = pd.Timestamp("2015-01-01")
test_start_60 = pd.Timestamp("2013-07-01")

def split(loans, test_start, name):
    train = loans[loans["issue_d"] < test_start].copy()
    test = loans[loans["issue_d"] >= test_start].copy()
    print(f"{name}:  train {len(train):,} loans (default rate {train['target'].mean():.1%}), "
          f"test {len(test):,} loans (default rate {test['target'].mean():.1%}), "
          f"train share {len(train) / len(loans):.0%}")
    return train, test

train_36, test_36 = split(loans_36, test_start_36, "36-month")
train_60, test_60 = split(loans_60, test_start_60, "60-month")

36-month:  train 335,558 loans (default rate 13.1%), test 283,026 loans (default rate 14.9%), train share 54%
60-month:  train 32,837 loans (default rate 25.4%), test 22,132 loans (default rate 24.8%), train share 60%


### 1.7 Saving the train and test sets

Only the features and the target are kept. The other columns were used to build the target and split the data, and they are not needed any more. The four data sets are saved so that the next notebook can start from here.

In [ ]:
# Folder for the cleaned data
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

# Payments are known only after a loan ends: saved apart, used only for profit (Section 10)
PAYMENTS = ["funded_amnt", "total_pymnt"]

train_36[PAYMENTS].to_parquet(PROCESSED / "payments_train_36.parquet", index=False)
test_36[PAYMENTS].to_parquet(PROCESSED / "payments_test_36.parquet", index=False)
train_60[PAYMENTS].to_parquet(PROCESSED / "payments_train_60.parquet", index=False)
test_60[PAYMENTS].to_parquet(PROCESSED / "payments_test_60.parquet", index=False)

# The 16 candidate features and the target
KEEP = [
    "purpose", "verification_status", "home_ownership",
    "annual_inc", "dti", "delinq_2yrs", "inq_last_6mths", "open_acc", "total_acc",
    "pub_rec", "revol_bal", "revol_util",
    "fico", "credit_history_years", "loan_to_income", "emp_length_years",
    "target",
]

train_36 = train_36[KEEP]
test_36 = test_36[KEEP]
train_60 = train_60[KEEP]
test_60 = test_60[KEEP]

# Save for the next notebooks
train_36.to_parquet(PROCESSED / "train_36.parquet", index=False)
test_36.to_parquet(PROCESSED / "test_36.parquet", index=False)
train_60.to_parquet(PROCESSED / "train_60.parquet", index=False)
test_60.to_parquet(PROCESSED / "test_60.parquet", index=False)